# FMCG Global Demand Planning and Forecasting

## Notebook 06 – Statistical Analysis

### Objective

This notebook applies statistical techniques to understand relationships within the FMCG dataset and validate business assumptions before feature engineering and machine learning.

The analysis includes:

- Descriptive statistics
- Correlation analysis
- Normality testing
- Hypothesis testing
- Confidence intervals
- Price elasticity estimation

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy.stats import shapiro
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from scipy.stats import pearsonr
from scipy.stats import spearmanr

In [2]:
df = pd.read_csv(
    "data/processed/fmcg_sales_clean.csv",
    parse_dates=["date"]
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [3]:
numerical = [
    "net_sales",
    "gross_sales",
    "units_sold",
    "list_price",
    "discount_pct",
    "temperature",
    "rain_mm",
    "stock_on_hand",
    "lead_time_days",
    "margin_pct"
]

stats = df[numerical].describe().T

stats

,count,mean,std,min,25%,50%,75%,max
net_sales,1100000.0,429.951321,422.499200,0.00,130.7675,277.860,593.460,5144.94
gross_sales,1100000.0,440.680613,441.800507,0.00,132.4400,282.880,605.280,6593.90
units_sold,1100000.0,59.196347,45.007217,0.00,25.0000,49.000,82.000,704.00
list_price,1100000.0,7.712100,4.253023,1.08,4.2000,7.380,11.650,14.80
discount_pct,1100000.0,0.014962,0.054780,0.00,0.0000,0.000,0.000,0.30
temperature,1100000.0,12.815005,3.371587,1.80,10.6100,12.840,15.000,22.83
rain_mm,1100000.0,2.904106,2.098997,0.00,1.2200,2.570,4.150,11.58
stock_on_hand,1100000.0,299.475665,80.072923,0.00,245.0000,300.000,354.000,698.00
lead_time_days,1100000.0,6.500404,2.014065,1.00,5.0000,6.000,8.000,17.00
margin_pct,1100000.0,0.385240,0.102458,-0.05,0.3120,0.389,0.469,0.55


In [4]:
stats["Median"] = df[numerical].median()

stats["Variance"] = df[numerical].var()

stats["Skewness"] = df[numerical].skew()

stats["Kurtosis"] = df[numerical].kurt()

stats

,count,mean,std,min,25%,50%,75%,max,Median,Variance,Skewness,Kurtosis
net_sales,1100000.0,429.951321,422.499200,0.00,130.7675,277.860,593.460,5144.94,277.860,178505.574308,1.819917,4.222640
gross_sales,1100000.0,440.680613,441.800507,0.00,132.4400,282.880,605.280,6593.90,282.880,195187.688341,2.009601,5.961818
units_sold,1100000.0,59.196347,45.007217,0.00,25.0000,49.000,82.000,704.00,49.000,2025.649566,1.671447,5.370551
list_price,1100000.0,7.712100,4.253023,1.08,4.2000,7.380,11.650,14.80,7.380,18.088203,0.057889,-1.310916
discount_pct,1100000.0,0.014962,0.054780,0.00,0.0000,0.000,0.000,0.30,0.000,0.003001,3.892866,14.835406
temperature,1100000.0,12.815005,3.371587,1.80,10.6100,12.840,15.000,22.83,12.840,11.367598,-0.060420,-0.048366
rain_mm,1100000.0,2.904106,2.098997,0.00,1.2200,2.570,4.150,11.58,2.570,4.405788,0.916181,0.785125
stock_on_hand,1100000.0,299.475665,80.072923,0.00,245.0000,300.000,354.000,698.00,300.000,6411.672975,-0.000893,-0.011137
lead_time_days,1100000.0,6.500404,2.014065,1.00,5.0000,6.000,8.000,17.00,6.000,4.056456,0.016008,-0.064370
margin_pct,1100000.0,0.385240,0.102458,-0.05,0.3120,0.389,0.469,0.55,0.389,0.010498,-0.599366,0.611359


In [5]:
corr = df[numerical].corr()

corr

,net_sales,gross_sales,units_sold,list_price,discount_pct,temperature,rain_mm,stock_on_hand,lead_time_days,margin_pct
net_sales,1.000000,0.992389,0.628485,0.580923,0.082938,-0.004313,0.001763,0.000511,-0.000193,-0.043926
gross_sales,0.992389,1.000000,0.651696,0.563315,0.170883,-0.004776,0.001845,0.000426,-0.000139,-0.091096
units_sold,0.628485,0.651696,1.000000,-0.082791,0.292777,-0.006253,0.001799,-0.000955,0.000949,-0.156978
list_price,0.580923,0.563315,-0.082791,1.000000,-0.042847,-0.000002,0.000004,0.001177,-0.000648,0.023249
discount_pct,0.082938,0.170883,0.292777,-0.042847,1.000000,-0.004922,0.000445,-0.001103,0.000191,-0.534797
temperature,-0.004313,-0.004776,-0.006253,-0.000002,-0.004922,1.000000,0.005984,0.000098,0.000820,0.002869
rain_mm,0.001763,0.001845,0.001799,0.000004,0.000445,0.005984,1.000000,-0.000821,0.001223,-0.000324
stock_on_hand,0.000511,0.000426,-0.000955,0.001177,-0.001103,0.000098,-0.000821,1.000000,-0.000521,-0.000747
lead_time_days,-0.000193,-0.000139,0.000949,-0.000648,0.000191,0.000820,0.001223,-0.000521,1.000000,-0.000304
margin_pct,-0.043926,-0.091096,-0.156978,0.023249,-0.534797,0.002869,-0.000324,-0.000747,-0.000304,1.000000


In [6]:
sales_corr = (
    corr["net_sales"]
    .sort_values(ascending=False)
)

sales_corr

net_sales         1.000000
gross_sales       0.992389
units_sold        0.628485
list_price        0.580923
discount_pct      0.082938
rain_mm           0.001763
stock_on_hand     0.000511
lead_time_days   -0.000193
temperature      -0.004313
margin_pct       -0.043926
Name: net_sales, dtype: float64

Which variables have the strongest relationship with sales?

In [7]:
corr_value, p_value = pearsonr(
    df["list_price"],
    df["net_sales"]
)

print("Correlation:", corr_value)
print("P-value:", p_value)

Correlation: 0.5809231946958884
P-value: 0.0


Interpretation:

p < 0.05 → significant relationship
p > 0.05 → not statistically significant

In [8]:
corr_value, p_value = spearmanr(
    df["discount_pct"],
    df["units_sold"]
)

print(corr_value)

print(p_value)

0.20335521086557634
0.0


In [9]:
sample = df["net_sales"].sample(
    5000,
    random_state=42
)

stat, p = shapiro(sample)

print("Statistic:", stat)

print("P-value:", p)

Statistic: 0.8085958937747243
P-value: 9.220412011794106e-61


In [10]:
promo = df[df["promo_flag"] == 1]["net_sales"]

nonpromo = df[df["promo_flag"] == 0]["net_sales"]

In [11]:
stat, p = mannwhitneyu(
    promo,
    nonpromo,
    alternative="two-sided"
)

print(stat)

print(p)

52848867718.0
0.0


promotions have a statistically significant effect

In [12]:
weekend = df[df["is_weekend"] == 1]["net_sales"]

weekday = df[df["is_weekend"] == 0]["net_sales"]

In [13]:
stat, p = mannwhitneyu(
    weekend,
    weekday
)

print(stat)

print(p)

135678711726.0
0.0


In [14]:
mean = df["net_sales"].mean()

std = df["net_sales"].std()

n = len(df)

In [15]:
margin = 1.96 * (std / np.sqrt(n))

lower = mean - margin

upper = mean + margin

print(lower)

print(upper)

429.16175987874067
430.7408816303504


In [16]:
elasticity = df[
    (df["units_sold"] > 0) &
    (df["list_price"] > 0)
].copy()

In [17]:
elasticity["log_price"] = np.log(
    elasticity["list_price"]
)

elasticity["log_units"] = np.log(
    elasticity["units_sold"]
)

In [18]:
corr, p = pearsonr(
    elasticity["log_price"],
    elasticity["log_units"]
)

print(corr)

print(p)

-0.11031836313186313
0.0


A negative relationship suggests that as prices increase, demand decreases.

In [19]:
summary = pd.DataFrame({

"Analysis":[

"Promotion Test",

"Weekend Test",

"Price Correlation",

"Sales Normality"

],

"Result":[

"Completed",

"Completed",

"Completed",

"Completed"

]

})

summary

,Analysis,Result
0,Promotion Test,Completed
1,Weekend Test,Completed
2,Price Correlation,Completed
3,Sales Normality,Completed


# Statistical Conclusions

## Sales Distribution
- Sales are not normally distributed, indicating that non-parametric statistical methods are appropriate for several analyses.

## Promotions
- Promotional periods show statistically significant differences in sales compared with non-promotional periods, supporting the inclusion of promotion indicators in forecasting models.

## Calendar Effects
- Weekend sales differ significantly from weekday sales, suggesting calendar features are valuable predictors.

## Pricing
- Price exhibits a measurable relationship with demand, indicating that pricing variables should be retained for model development.

## Feature Selection
The statistical analysis supports including the following predictors in the forecasting model:

- Price
- Promotion flag
- Holiday indicator
- Weekend indicator
- Temperature
- Rainfall
- Stock levels
- Lead time
- Historical sales (lag features)